In [1]:
import os
os.chdir('../../../../..')

In [2]:
import numpy as np

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN, KMeans
from kmedoids import KMedoids

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [3]:
qm9 = QM9Dataset(limit=5000, sampling_strategy="stratified", stratify_by=["num_atoms", "gap"], add_selfies_transformer=True)
df = qm9.load()

2026-04-21 11:06:33.873 | INFO     | src.datasets:load:496 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-21 11:06:34.207 | INFO     | src.datasets:_sample_qm9_df:688 - QM9 sampling complete: strategy=stratified, requested_limit=5000, returned_rows=5000.
2026-04-21 11:06:34.208 | INFO     | src.datasets:_add_requested_descriptors:129 - Applying requested QM9 descriptors to sampled dataframe (rows=5000).
2026-04-21 11:06:34.208 | INFO     | src.features:compute_selfies_transformer:93 - Computing SELFormer Embeddings using HUBioDataLab/SELFormer...
INFO: HTTP Request: HEAD https://huggingface.co/HUBioDataLab/SELFormer/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/HUBioDataLab/SELFormer/177d98b158e999a6cb7fc9743dbfe1e8a17c57e5/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/HUBioDataLab/SELFormer/resolve/main/tokenizer_config.json "HTTP/1.1 

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

INFO: HTTP Request: GET https://huggingface.co/api/models/HUBioDataLab/SELFormer "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/HUBioDataLab/SELFormer/commits/main "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/HUBioDataLab/SELFormer/discussions?p=0 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/HUBioDataLab/SELFormer/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/HUBioDataLab/SELFormer/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/HUBioDataLab/SELFormer/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
2026-04-21 11:07:43.479 | INFO     | src.datasets:_add_requested_descriptors:152 - Added descriptor column(s): ['selfies_transformer']


In [4]:
len(df['selfies_transformer'].to_list()[0])

768

In [5]:
molecules = qm9.get_molecules()

2026-04-21 11:08:19.177 | SUCCESS  | src.datasets:get_molecules:1181 - Saved 5000 molecules to data/QM9/qm9_subset.xyz (failed: 0, requested: 5000).


In [6]:
plot_molecules_with_py3dmol(molecules[0:3])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
dist_matrix = qm9.get_distance_matrix(
    descriptor="transformer",
    dist_type="euclidean",
    force_calculate=True,
    pca_variance=60,
)

2026-04-21 11:10:11.009 | INFO     | src.datasets:get_distance_matrix:991 - Applying PCA to retain 60.00% of variance.
2026-04-21 11:10:11.536 | INFO     | src.datasets:get_distance_matrix:1001 - PCA reduced 'selfies_transformer' dimensions from 768 to 5
2026-04-21 11:10:11.549 | INFO     | src.datasets:get_distance_matrix:1012 - Calculating distance matrix for selfies_transformer using euclidean distance.
2026-04-21 11:10:11.747 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/QM9/dist_selfies_transformer_euclidean_pca0.6.npy


# Determining the best number of clusters for each clustering method

In [11]:
out = evaluate_distance_matrix_clustering_sweep(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="euclidean",
    dataset_name="qm9",
)

Evaluation k:   0%|          | 0/19 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [12]:
# find the n molecules that are not on the diagonal with the smallest distance
n = 10
# Get the indices of the upper triangle (excluding diagonal)
triu_indices = np.triu_indices_from(dist_matrix, k=1)
# Get the distances and corresponding molecule pairs
distances = dist_matrix[triu_indices]
molecule_pairs = list(zip(triu_indices[0], triu_indices[1]))
# Get the indices of the n smallest distances
smallest_indices = np.argsort(distances)[:n]
# Get the corresponding molecule pairs for the n smallest distances
closest_pairs = [molecule_pairs[i] for i in smallest_indices]
print("Closest molecule pairs (indices):", closest_pairs)
mols = [(molecules[idx1], molecules[idx2]) for idx1, idx2 in closest_pairs]

Closest molecule pairs (indices): [(np.int64(4966), np.int64(4967)), (np.int64(1021), np.int64(2358)), (np.int64(2809), np.int64(3979)), (np.int64(4140), np.int64(4201)), (np.int64(2866), np.int64(3237)), (np.int64(4933), np.int64(4958)), (np.int64(2621), np.int64(3375)), (np.int64(3848), np.int64(4607)), (np.int64(848), np.int64(923)), (np.int64(1373), np.int64(4256))]


In [11]:
print(mols[0])

(Atoms(symbols='N2HFC2HC2NC', pbc=False, initial_charges=..., mass=..., partial_charge=...), Atoms(symbols='CNHNCFCHC2N', pbc=False, initial_charges=..., mass=..., partial_charge=...))


In [15]:
plot_molecules_with_py3dmol(mols[1])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Hiercical Clustering on Distance Matrix

In [ ]:
model_hier = AgglomerativeClustering(metric='precomputed', n_clusters=3, linkage='complete')
labels_hier = model_hier.fit_predict(dist_matrix)
print(np.unique(labels_hier, return_counts=True))
df = df.with_columns(labels_hier=labels_hier)

(array([0, 1, 2]), array([4865,    2,  133]))


In [22]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [23]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_hier,
    clustering_method="hierarchical"
)

2026-04-21 11:13:06.287 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/transformer/pca_hierarchical_projection.png


{'coords': array([[  2.3214721 , -58.46608817],
        [ 44.20252122,  61.41668055],
        [ 72.14163397, -58.30720516],
        ...,
        [  7.31243128,  30.99989679],
        [ 15.32269402, -18.02808763],
        [ 15.34732849, -44.20419622]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/transformer/pca_hierarchical_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/transformer'),
 'clustering_method': 'hierarchical'}

In [24]:
average_numeric_by_cluster(df, "labels_hier")

shape: (3, 59)
┌─────────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬────────────────────┬──────────────┬─────────────┐
│ labels_hier ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election_affinity ┆ ionization_energies ┆ num_heavy_atoms ┆ num_ring

labels_hier,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,4865,2.092139,18.169373,122.876876,0.098869,35.949024,0.86518,12.848367,8.789723,1.651799,0.161768,2.061043,2.334841,0.064827,0.219404,0.715769,0.914697,1.9926,6.548201,0.428571,1.263104,4.686742,6.362384,38.284481,1.26233,0.002878,0.344707,0.030627,0.127235,0.126619,0.00185,0.040288,0.124974,0.529702,0.002878,2.411305,2.678711,75.387054,-6.535024,0.3483,6.883329,1197.150132,4.100373,-11163.632986,-11163.400472,-11163.374797,-11164.544524,31.824055,-76.618208,-77.085968,-77.527257,-71.302706,3.38676,1.392769,1.115361,73.627955,15.786228,10.585817
1,2,2.307692,13.0,121.0,-2.0,45.0,0.714629,12.731444,9.0,3.0,0.0,2.285714,0.0,0.166667,0.166667,0.666667,1.0,2.0,6.0,1.0,1.0,4.0,5.0,30.0,1.301325,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,3.9714,68.260002,-7.517145,-1.249003,6.268143,991.86496,2.449201,-11308.250488,-11308.066406,-11308.040527,-11309.103516,25.623,-61.403234,-61.719854,-62.028212,-57.496889,3.788615,1.45969,1.176935,100.0,0.0,0.0
2,133,1.742798,12.962406,119.744361,0.06015,66.646617,0.490075,13.094942,8.631579,1.203008,0.894737,2.03147,1.285714,0.053777,0.839885,0.106337,1.037594,3.180451,5.082707,0.263158,3.533835,0.473684,6.030075,22.736842,1.267869,0.007519,0.090226,0.142857,0.24812,0.120301,0.0,0.007519,0.030075,0.180451,0.007519,4.360902,3.202626,67.263384,-6.561115,-1.38866,5.172537,998.06838,2.519031,-11778.743986,-11778.553087,-11778.527417,-11779.606776,25.958632,-58.09137,-58.400095,-58.707494,-54.227374,4.092664,1.701421,1.183896,24.06015,70.676692,5.263158


# KMedoids

In [25]:
model_km = KMedoids(n_clusters=3, metric="precomputed")
labels_km = model_km.fit_predict(dist_matrix)
print(np.unique(labels_km, return_counts=True))
df = df.with_columns(labels_km=labels_km)

(array([0, 1, 2], dtype=uint64), array([1360, 1443, 2197]))


In [26]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [27]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_km,
    clustering_method="kmedoids"
)

2026-04-21 11:16:40.276 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/transformer/pca_kmedoids_projection.png


{'coords': array([[  2.3214721 , -58.46608817],
        [ 44.20252122,  61.41668055],
        [ 72.14163397, -58.30720516],
        ...,
        [  7.31243128,  30.99989679],
        [ 15.32269402, -18.02808763],
        [ 15.34732849, -44.20419622]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/transformer/pca_kmedoids_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/transformer'),
 'clustering_method': 'kmedoids'}

In [28]:
average_numeric_by_cluster(df, "labels_km")

shape: (3, 60)
┌───────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬────────────────────┬──────────────┬─────────────┐
│ labels_km ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election_affinity ┆ ionization_energies ┆ num_heavy_atoms 

labels_km,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,pct_aliphatic_ring,pct_aromatic,pct_acyclic
u64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,1360,2.158816,17.573529,121.880882,-0.154412,33.028676,0.899248,12.787858,8.780882,2.642647,0.027941,2.160157,1.468382,0.053559,0.111084,0.835357,1.005882,1.873529,6.740441,0.356618,0.686029,5.509559,6.002206,38.066176,1.278244,0.0,0.469853,0.002206,0.055147,0.045588,0.000735,0.033824,0.111765,0.665441,0.0,2.228676,2.453552,73.20239,-6.516709,0.656851,7.17359,1040.947403,3.936571,-11055.185498,-11054.977508,-11054.951825,-11056.059873,29.681021,-74.585847,-75.055177,-75.481144,-69.40925,3.270598,1.621437,1.329464,0.038235,96.838235,2.426471,0.735294
1,1443,1.89637,15.550243,122.030492,-0.025641,52.538462,0.739832,12.942847,8.731809,1.003465,0.557173,2.000328,2.227304,0.085493,0.526333,0.388174,1.082467,2.769924,5.701317,0.513514,2.665281,2.092862,6.395703,29.700624,1.257587,0.00693,0.196812,0.112266,0.260568,0.239085,0.002079,0.057519,0.112959,0.335412,0.010395,3.460152,3.237639,71.213576,-6.451759,-0.581437,5.870301,1185.428298,3.282044,-11651.292414,-11651.065629,-11651.039955,-11652.205285,30.011294,-67.630611,-68.003143,-68.377083,-63.068353,3.801959,1.375746,1.024397,0.149688,30.35343,53.014553,16.632017
2,2197,2.158494,19.938553,123.857988,0.333182,28.727355,0.903574,12.83859,8.823851,1.438325,0.029131,2.037982,2.876195,0.057653,0.122378,0.819969,0.755576,1.627674,6.896222,0.407829,0.836595,5.625398,6.542103,43.108785,1.255966,0.002276,0.348657,0.001365,0.091488,0.102412,0.002276,0.030951,0.136095,0.551661,0.0,1.954028,2.483877,78.982308,-6.603523,0.661351,7.264885,1289.30392,4.642019,-10947.836458,-10947.587562,-10947.561893,-10948.767121,33.980558,-82.643977,-83.163547,-83.650329,-76.836908,3.229062,1.281143,1.046775,0.0,84.706418,2.913063,12.380519


# Spectral

In [29]:
model_spectral = SpectralClustering(
                n_clusters=3,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)

In [30]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [33]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_spectral,
    clustering_method="spectral"
)

2026-04-21 11:23:09.020 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/transformer/pca_spectral_projection.png


{'coords': array([[  2.3214721 , -58.46608817],
        [ 44.20252122,  61.41668055],
        [ 72.14163397, -58.30720516],
        ...,
        [  7.31243128,  30.99989679],
        [ 15.32269402, -18.02808763],
        [ 15.34732849, -44.20419622]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/transformer/pca_spectral_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/transformer'),
 'clustering_method': 'spectral'}

In [35]:
average_numeric_by_cluster(df, "labels_spectral")

shape: (3, 61)
┌─────────────────┬───────┬─────────────────────┬───────────┬────────────┬────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────┬────────────────────┬──────────────┬─────────────┐
│ labels_spectral ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp   ┆ tpsa    ┆ election_affinity ┆ ionization_energ

labels_spectral,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,labels_km,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,4998,2.082936,18.027811,122.792117,0.096639,36.770308,0.855168,12.854874,8.785514,1.640856,0.181072,2.060391,2.305122,0.064571,0.235868,0.699561,0.918167,2.02441,6.508804,0.42437,1.323129,4.57443,6.352741,37.865746,1.262501,0.003001,0.337935,0.033613,0.130452,0.126451,0.001801,0.039416,0.122849,0.520408,0.003001,2.463585,2.692963,75.164892,-6.536165,0.30156,6.837732,1191.541761,4.057355,-11180.24451,-11180.013137,-11179.987463,-11181.154697,31.664068,-76.115321,-76.578768,-77.016417,-70.839257,3.405351,1.401275,1.11739,0.053621,1.167067,72.34894,17.226891,10.42417
1,1,2.1,20.0,125.0,1.0,40.0,0.859605,12.827849,9.0,0.0,0.0,1.9,5.0,0.142857,0.142857,0.714286,0.0,2.0,6.0,1.0,1.0,5.0,7.0,42.0,1.245634,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.691,81.389999,-6.990605,-0.829947,6.157937,2057.528809,4.582533,-10971.134766,-10970.84375,-10970.818359,-10972.160156,36.301998,-84.434273,-84.914497,-85.402863,-78.684479,4.50213,0.55228,0.51403,0.0,2.0,0.0,0.0,100.0
2,1,2.047619,21.0,124.0,1.0,28.0,0.718342,12.908166,9.0,1.0,1.0,2.0,4.0,0.0,0.428571,0.571429,1.0,1.0,8.0,0.0,3.0,4.0,7.0,43.0,1.239401,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,3.7011,85.050003,-5.809631,0.925187,6.737539,1478.9646,5.001507,-10430.735352,-10430.483398,-10430.458008,-10431.680664,34.456001,-87.730934,-88.288414,-88.80249,-81.608589,4.04218,0.90265,0.8134,0.0,2.0,0.0,100.0,0.0


# DBSCAN 

In [40]:
model_db = DBSCAN(
    eps=2,
    min_samples=3,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)
print(np.unique(labels_db,return_counts=True))

(array([-1,  0,  1,  2]), array([  49, 4945,    3,    3]))


In [41]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [44]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="transformer",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_db,
    clustering_method="dbscan"
)

2026-04-21 11:33:20.092 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/transformer/pca_dbscan_projection.png


{'coords': array([[  2.3214721 , -58.46608817],
        [ 44.20252122,  61.41668055],
        [ 72.14163397, -58.30720516],
        ...,
        [  7.31243128,  30.99989679],
        [ 15.32269402, -18.02808763],
        [ 15.34732849, -44.20419622]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/transformer/pca_dbscan_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/transformer'),
 'clustering_method': 'dbscan'}

In [43]:
average_numeric_by_cluster(df, "labels_db")

shape: (4, 62)
┌───────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────┬─────────────────┬────────────────────┬──────────────┬─────────────┐
│ labels_db ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election_affinity ┆ ionizati

labels_db,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,labels_km,labels_spectral,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-1,49,1.965449,13.755102,117.346939,-0.387755,53.285714,0.589132,13.006415,8.510204,1.816327,0.367347,2.09624,0.979592,0.120165,0.458698,0.421137,1.102041,2.469388,5.367347,0.673469,1.836735,2.469388,5.734694,27.55102,1.274557,0.020408,0.102041,0.102041,0.163265,0.061224,0.0,0.0,0.081633,0.163265,0.0,3.530612,3.412837,67.085102,-6.82717,-0.572938,6.254343,947.960518,2.769427,-11130.095554,-11129.902872,-11129.877172,-11130.95442,26.502694,-60.605716,-60.943286,-61.27107,-56.531379,3.772621,1.806555,1.331326,0.938776,0.408163,0.0,67.346939,28.571429,4.081633
0,4945,2.084403,18.072599,122.846714,0.10091,36.608898,0.857777,12.853516,8.788069,1.639434,0.178362,2.060064,2.320121,0.064044,0.232987,0.702969,0.915875,2.02002,6.519515,0.422042,1.313246,4.599798,6.358746,37.978362,1.262399,0.002831,0.339939,0.032963,0.130233,0.127199,0.00182,0.039838,0.123357,0.524368,0.003033,2.452983,2.686467,75.242487,-6.534372,0.31072,6.845098,1194.222125,4.070866,-11180.569422,-11180.337631,-11180.311956,-11181.480172,31.718379,-76.27566,-76.740414,-77.179215,-70.987052,3.402472,1.396925,1.115204,0.042467,1.17634,0.000607,72.457027,17.0273,10.515672
1,3,1.845316,17.333333,119.666667,1.333333,23.666667,0.760024,12.700914,9.0,1.0,1.0,2.0,0.0,0.0,1.0,0.0,1.0,0.666667,8.333333,0.0,7.333333,0.0,6.0,32.0,1.238992,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.666667,2.032567,89.093333,-5.396925,-0.965097,4.430921,1050.366536,3.873985,-10217.553385,-10217.349284,-10217.323893,-10218.41569,29.755,-76.276861,-76.741259,-77.161031,-71.139061,2.515897,1.82187,1.1564,2.0,0.0,0.0,0.0,100.0,0.0
2,3,1.816176,16.333333,126.0,0.333333,44.333333,0.951667,12.780598,9.0,1.0,1.0,2.0,3.0,0.095238,0.612698,0.292063,1.333333,3.0,6.0,0.666667,3.666667,1.666667,7.333333,29.666667,1.244771,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,2.639233,70.673332,-5.786955,0.583231,6.371092,1277.547404,3.495883,-12106.931966,-12106.704753,-12106.679362,-12107.850911,30.830667,-71.629382,-72.031677,-72.425743,-66.825147,3.619507,1.08285,0.88438,2.0,0.0,0.0,0.0,100.0,0.0


# KMeans on Raw Embeddings


In [47]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

X_raw = np.array(df["selfies_transformer"].to_list(), dtype=np.float32)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_raw)

kmeans_raw = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans_raw = kmeans_raw.fit_predict(X_pca)
df = df.with_columns(labels_kmeans_raw=labels_kmeans_raw)


In [48]:
average_numeric_by_cluster(df, "labels_kmeans_raw")

shape: (3, 63)
┌───────────────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────┐
│ labels_kmeans_raw ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ 

labels_kmeans_raw,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,labels_km,labels_spectral,labels_db,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,1121,1.853263,14.958073,121.569135,-0.027654,55.069581,0.719895,12.952989,8.707404,0.988403,0.613738,1.999086,2.098127,0.100823,0.564388,0.33479,1.099019,2.906334,5.424621,0.599465,2.750223,1.752899,6.413024,27.899197,1.258569,0.006244,0.197145,0.134701,0.275647,0.187333,0.000892,0.0562,0.087422,0.362177,0.013381,3.604817,3.235375,70.188724,-6.48944,-0.680989,5.808429,1169.878832,3.101414,-11707.613811,-11707.392052,-11707.366402,-11708.520513,29.343079,-65.524888,-65.879629,-66.238343,-61.128885,3.940602,1.400183,1.031162,0.223015,0.986619,0.0,-0.014273,25.691347,57.448707,16.859946
1,2483,2.138521,19.521547,123.847362,0.275876,31.334676,0.88778,12.84983,8.824406,1.371325,0.078534,2.032403,2.830447,0.05552,0.16613,0.778349,0.782521,1.74708,6.82642,0.387032,1.075715,5.234797,6.509062,41.851792,1.255932,0.003222,0.318566,0.006444,0.110753,0.144986,0.002819,0.036649,0.150222,0.516714,0.0,2.126863,2.62554,78.20553,-6.573573,0.489222,7.062808,1280.038713,4.510405,-11041.572555,-11041.324729,-11041.299051,-11042.503444,33.668489,-81.267494,-81.772062,-82.248121,-75.579267,3.24552,1.283981,1.04045,0.0,1.829642,0.001208,-0.002014,79.097865,7.853403,13.048731
2,1396,2.168486,17.839542,121.899713,-0.12106,31.739971,0.905693,12.785077,8.77937,2.64255,0.016476,2.159241,1.540115,0.051569,0.096172,0.852258,1.01361,1.808739,6.815186,0.350287,0.618195,5.665473,6.027221,38.785817,1.277315,0.0,0.484957,0.000716,0.048711,0.044413,0.000716,0.030802,0.102436,0.653295,0.0,2.145415,2.378045,73.764097,-6.506956,0.756404,7.26338,1052.358109,4.020217,-11002.725066,-11002.515183,-11002.489494,-11003.601341,29.967995,-75.46988,-75.947559,-76.380367,-70.21927,3.261064,1.609812,1.32283,0.012894,0.13467,0.0,-0.01361,97.707736,1.647564,0.644699


In [32]:

create_chemiscope_viewer(df, X_raw, labels_kmeans_raw, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…